# 2. Ball Tracking — Object Detection

Trains a YOLO11n model to detect the volleyball in each frame.
Detections are post-processed with a Kalman filter in the backend to fill missed frames.

| | |
|---|---|
| **Dataset** | [volleyball_v2 v3](https://universe.roboflow.com/shukur-sabzaliev1/volleyball_v2/dataset/3) |
| **Model** | `yolo11n.pt` |
| **Task** | Object detection (single class: volleyball) |
| **Output** | `runs/detect/ball-tracking-v3/weights/best.pt` |

> **Why `imgsz=1280`?** The volleyball is a small object that often occupies <1% of frame
> area. Higher resolution gives the model much more pixels to work with, significantly
> improving detection recall. Batch is halved to compensate for the increased memory usage.

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────

# Model
MODEL = "yolo11n.pt"       # nano is sufficient; speed matters for a single-class detector

# Training
EPOCHS      = 150
IMGSZ       = 1280          # higher res is critical for small-object (ball) detection
BATCH       = 8             # reduced from 16 to fit larger imgsz on RTX 3060
LR0         = 0.01
LRF         = 0.005
COS_LR      = True
COPY_PASTE  = 0.1           # augmentation: pastes extra ball instances to improve recall
FREEZE      = 0             # freeze first N backbone layers (0 = off; try 10 for small datasets)
PATIENCE    = 20
SAVE_PERIOD = 10
WORKERS     = 4

# Data
DATASET_DIR  = "datasets/ball_tracking"
RF_WORKSPACE = "shukur-sabzaliev1"
RF_PROJECT   = "volleyball_v2"
RF_VERSION   = 3
RF_FORMAT    = "yolov11"

# Run identity
RUN_PROJECT   = "runs/detect"
RUN_NAME      = "ball-tracking-v3"
WANDB_PROJECT = "tobio"

In [ ]:
import os
import wandb
from dotenv import load_dotenv
from roboflow import Roboflow
from ultralytics import YOLO

from utils import setup_device, print_metrics

load_dotenv()
DEVICE = setup_device()

if WANDB_PROJECT:
    wandb.init(
        project=WANDB_PROJECT,
        name=RUN_NAME,
        config={
            "model": MODEL, "epochs": EPOCHS, "imgsz": IMGSZ,
            "batch": BATCH, "lr0": LR0, "lrf": LRF, "cos_lr": COS_LR,
            "copy_paste": COPY_PASTE, "freeze": FREEZE,
            "dataset": f"{RF_PROJECT} v{RF_VERSION}",
        },
    )

In [ ]:
rf = Roboflow(api_key=os.getenv("ROBOFLOW_API_KEY"))
dataset = (
    rf.workspace(RF_WORKSPACE)
    .project(RF_PROJECT)
    .version(RF_VERSION)
    .download(RF_FORMAT, location=DATASET_DIR)
)
print(f"Dataset location: {dataset.location}")

In [ ]:
model = YOLO(MODEL)

train_results = model.train(
    data=f"{DATASET_DIR}/data.yaml",
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    optimizer="Adam",
    lr0=LR0,
    lrf=LRF,
    cos_lr=COS_LR,
    copy_paste=COPY_PASTE,
    freeze=FREEZE,
    patience=PATIENCE,
    project=RUN_PROJECT,
    name=RUN_NAME,
    save=True,
    save_period=SAVE_PERIOD,
    exist_ok=True,
    verbose=True,
    plots=True,
    val=True,
)

In [ ]:
# augment=True enables Test-Time Augmentation (TTA): multi-scale + flip ensemble
metrics = model.val(augment=True)
print_metrics(metrics)

# tensorboard --logdir runs/detect

In [ ]:
onnx_path = model.export(format="onnx")
print(f"ONNX model saved to: {onnx_path}")

if WANDB_PROJECT:
    wandb.finish()